# Heliophysics Phenomena Harmonization

**Purpose:** Assemble a unified, sourced phenomena dictionary for the Heliophysics domain.

We treat a *phenomenon* broadly: any temporally and/or spatially extended process, event, or quality that occurs in the heliophysics domain. This includes:
- Discrete events (flares, CMEs, substorms, geomagnetic storms)
- Ongoing processes (reconnection, ionization, particle precipitation, Joule heating)
- Features/structures that emerge from processes (aurora types, ionization patches, current systems)
- Dynamical qualities (wave-particle interactions, momentum transfer, ion drag)

**Inputs:**
- HelioKNOW phenomena ontology (`hk_phenomenon.ttl` + `hk_core.ttl`)
- Standard heliophysics/space-physics glossaries (HELIO Ontology, HEK, NASA CCMC, SWEET, SPASE, NASA Heliophysics Vocabulary, Space Weather Glossaries, ESA, UAT, AGU, GCMD, GEMET)

**Outputs:**
- `phenomena_longform.csv`: one row per (phenomenon_canonical, source, definition)
- `phenomena_unified.csv`: one row per canonical phenomenon with all definitions and sources compiled
- `phenomena_helioknow_coverage.csv`: HelioKNOW phenomena with cross-source coverage summary

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
import os

# Standard glossaries (JSON)
GLOSSARIES_LIST = [
    '/Users/ryanmc/Documents/NASA_FDL/2022/sdm-kg-nasa-2022/data/raw/SMDVocabulary/SupersetVocab/HELIO Ontology.json',
    '/Users/ryanmc/Documents/NASA_FDL/2022/sdm-kg-nasa-2022/data/raw/SMDVocabulary/SupersetVocab/Heliophysics Events Knowledgebase.json',
    '/Users/ryanmc/Documents/NASA_FDL/2022/sdm-kg-nasa-2022/data/raw/SMDVocabulary/SupersetVocab/NASA CCMC.json',
    '/Users/ryanmc/Documents/NASA_FDL/2022/sdm-kg-nasa-2022/data/raw/SMDVocabulary/SupersetVocab/Semantic Web for Earth and Environment Technology Ontology.json',
    '/Users/ryanmc/Documents/NASA_FDL/2022/sdm-kg-nasa-2022/data/raw/SMDVocabulary/SupersetVocab/SPASE Dictionary.json',
    '/Users/ryanmc/Documents/NASA_FDL/2022/sdm-kg-nasa-2022/data/raw/SMDVocabulary/SupersetVocab/NASA Heliophysics Vocabulary.json',
    '/Users/ryanmc/Documents/NASA_FDL/2022/sdm-kg-nasa-2022/data/raw/SMDVocabulary/SupersetVocab/Space Weather Glossary.json',
    '/Users/ryanmc/Documents/NASA_FDL/2022/sdm-kg-nasa-2022/data/raw/SMDVocabulary/SupersetVocab/Space Weather Glossary (Second).json',
    '/Users/ryanmc/Documents/NASA_FDL/2022/sdm-kg-nasa-2022/data/raw/SMDVocabulary/SupersetVocab/ESA Space Weather Glossary.json',
    '/Users/ryanmc/Documents/NASA_FDL/2022/sdm-kg-nasa-2022/data/raw/SMDVocabulary/SupersetVocab/Unified Astronomy Thesaurus (UAT).json',
    '/Users/ryanmc/Documents/NASA_FDL/2022/sdm-kg-nasa-2022/data/raw/SMDVocabulary/SupersetVocab/AGU Index Terms.json',
    '/Users/ryanmc/Documents/NASA_FDL/2022/sdm-kg-nasa-2022/data/raw/SMDVocabulary/SupersetVocab/Global Change Master Directory (GCMD).json',
]

# GEMET RDF files
GEMET_PATHS = [
    '/Users/ryanmc/Documents/Helio_ECIP/dev/Helio-KNOW/tools/glossary_harmonization/data/gemet-backbone.rdf',
    '/Users/ryanmc/Documents/Helio_ECIP/dev/Helio-KNOW/tools/glossary_harmonization/data/gemet-definitions.rdf',
    '/Users/ryanmc/Documents/Helio_ECIP/dev/Helio-KNOW/tools/glossary_harmonization/data/gemet-skoscore.rdf',
]

# GCMD
GCMD_PATHS = [
    '/Users/ryanmc/Documents/Helio_ECIP/dev/Helio-KNOW/tools/glossary_harmonization/data/GCMD_sciencekeywords.csv',
    '/Users/ryanmc/Documents/Helio_ECIP/dev/Helio-KNOW/tools/glossary_harmonization/data/GCMDsciencekeywords_CMR_EarthData_page1.json',
    '/Users/ryanmc/Documents/Helio_ECIP/dev/Helio-KNOW/tools/glossary_harmonization/data/GCMDsciencekeywords_CMR_EarthData_page2.json',
]

# SWEET
SWEET_RDF_PATH = '/Users/ryanmc/Documents/Helio_ECIP/dev/Helio-KNOW/tools/glossary_harmonization/data/sweetAll.rdf'
SWEET_PHEN_MODULES = [
    'phenHelio', 'phenStar.ttl', 'phenAtmo.ttl', 'phenWave.ttl',
    'phenSystemComplexity.ttl', 'procWave.ttl', 'phenElecMag',
    'procPhysical', 'propEnergyFlux', 'statePhysical',
]

# UAT
UAT_HIERARCHY_CSV = '/Users/ryanmc/Documents/Helio_ECIP/dev/Helio-KNOW/tools/glossary_harmonization/data/UAT_Heliophysics.csv'
UAT_DEFINITIONS_JSON = '/Users/ryanmc/Documents/Helio_ECIP/dev/Helio-KNOW/tools/glossary_harmonization/data/UAT.json'

# HelioKNOW
HK_CORE_TTL   = '/Users/ryanmc/Documents/Helio_ECIP/dev/Helio-KNOW/data-models/hk_core.ttl'
HK_PHENOM_TTL = '/Users/ryanmc/Documents/Helio_ECIP/dev/Helio-KNOW/data-models/hk_phenomenon.ttl'

# Output directory
OUTPUT_DIR = '/Users/ryanmc/Documents/Helio_ECIP/dev/Helio-KNOW/tools/glossary_harmonization/data'

# Phenomenon type keywords used when filtering glossary entries
PHEN_HIERARCHY_KEYWORDS = {
    'phenomenon', 'phenomena', 'event', 'process', 'processes',
    'dynamics', 'interaction', 'emission', 'ejection', 'storm',
    'aurora', 'auroral', 'reconnection', 'precipitation', 'heating',
    'ionization', 'radiation', 'oscillation', 'wave', 'substorm',
    'flare', 'eruption', 'disturbance', 'shock', 'burst', 'current',
    'wind', 'flux', 'outflow', 'transport', 'turbulence', 'convection',
}



In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import re
import csv
import json
import time
from pathlib import Path
from functools import lru_cache

import numpy as np
import pandas as pd
from rdflib import Graph, Namespace, RDF, RDFS, OWL, URIRef, Literal
from rdflib.namespace import SKOS, DCTERMS



In [ ]:
# ── Shared helpers (reused from add_definitions_to_terms.ipynb) ────────────────

def _clean_text(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, str):
        value = re.sub(r'\s+', ' ', value).strip()
        if value.lower() in {'nan', 'none', 'null', 'n/a', 'na', ''}:
            return None
        return value
    return _clean_text(str(value))


def _normalize_term(value):
    value = _clean_text(value)
    if not value:
        return None
    # Split CamelCase
    split_camel = re.sub(r'([a-z](?=[A-Z])|[A-Z](?=[A-Z][a-z]))', r'\1 ', value)
    return split_camel.lower().strip()


def _first_nonempty(*values):
    for value in values:
        cleaned = _clean_text(value)
        if cleaned:
            return cleaned
    return None


def _stringify_hierarchy(hierarchy_values, term=None):
    items = [_clean_text(item) for item in hierarchy_values if _clean_text(item)]
    if term and (not items or items[-1].lower() != term.lower()):
        items = items + [term]
    if not items:
        return None
    return ' -> '.join(items)


def _safe_read_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def _unique_nonempty_list(series):
    values = []
    seen = set()
    for value in series:
        cleaned = _clean_text(value)
        if not cleaned:
            continue
        if cleaned not in seen:
            seen.add(cleaned)
            values.append(cleaned)
    return values


def _contains_phen_keyword(text):
    """Return True if text contains any phenomenon-indicative keyword."""
    t = str(text).lower()
    return any(k in t for k in PHEN_HIERARCHY_KEYWORDS)




## 1  Parse HelioKNOW Phenomena Ontology

Extract all instances of `hk:Phenomenon` (and `hkp:PrincipalPhenomenon`) from the two TTL files.  
Capture: `skos:prefLabel`, `skos:altLabel`, `skos:definition`, broader/narrower hierarchy, `hk:occursIn` regions, `hk:describedBy` disciplines.

In [ ]:
# ── Parse HelioKNOW TTL files ──────────────────────────────────────────────────

HK_NS   = Namespace('https://www.helioknowledge.network/core/')
HKP_NS  = Namespace('https://www.helioknowledge.network/phenomenon/')
HKR_NS  = Namespace('https://www.helioknowledge.network/region/')

hk_graph = Graph()

# Parse core first, then phenomenon (phenomenon imports core)
# We parse locally only — no remote imports fetched
print(f'Parsing {HK_CORE_TTL} ...')
hk_graph.parse(HK_CORE_TTL, format='turtle')
print(f'  Triples after core: {len(hk_graph):,}')

print(f'Parsing {HK_PHENOM_TTL} ...')
hk_graph.parse(HK_PHENOM_TTL, format='turtle')
print(f'  Triples after phenomenon: {len(hk_graph):,}')

# Identify the Phenomenon class node
HK_PHENOMENON_CLASS = HK_NS.Phenomenon
HK_PRINCIPAL_CLASS  = HKP_NS.PrincipalPhenomenon

def _literal_en(values):
    """Pick English or language-neutral literal from a list."""
    values = list(values)
    en = [v for v in values if isinstance(v, Literal) and v.language in ('en', None)]
    chosen = en[0] if en else (values[0] if values else None)
    return _clean_text(str(chosen)) if chosen is not None else None


def _all_literals_en(values):
    """Return list of English/neutral literals as clean strings."""
    out = []
    seen = set()
    for v in values:
        if isinstance(v, Literal) and v.language in ('en', None):
            t = _clean_text(str(v))
            if t and t not in seen:
                seen.add(t)
                out.append(t)
    return out


def _label_for_uri(uri):
    """Get best human label for a URI from the graph."""
    pref = list(hk_graph.objects(uri, SKOS.prefLabel))
    lbl  = list(hk_graph.objects(uri, RDFS.label))
    choices = pref + lbl
    return _literal_en(choices) or str(uri).rsplit('/', 1)[-1].rsplit('#', 1)[-1]


def _is_phenomenon_instance(node):
    """True if node is typed as hk:Phenomenon or hkp:PrincipalPhenomenon."""
    types = set(hk_graph.objects(node, RDF.type))
    return bool(types & {HK_PHENOMENON_CLASS, HK_PRINCIPAL_CLASS})


# Collect all phenomenon URIs
phen_uris = set()
for subj in hk_graph.subjects(RDF.type, HK_PHENOMENON_CLASS):
    if isinstance(subj, URIRef):
        phen_uris.add(subj)
for subj in hk_graph.subjects(RDF.type, HK_PRINCIPAL_CLASS):
    if isinstance(subj, URIRef):
        phen_uris.add(subj)

print(f'\nHelioKNOW phenomenon instances found: {len(phen_uris)}')

In [ ]:
# ── Build HelioKNOW phenomena DataFrame ───────────────────────────────────────

rows_hk = []

for uri in sorted(phen_uris, key=str):
    pref_label = _literal_en(hk_graph.objects(uri, SKOS.prefLabel))
    alt_labels = _all_literals_en(hk_graph.objects(uri, SKOS.altLabel))
    definition = _literal_en(hk_graph.objects(uri, SKOS.definition))
    related    = [_label_for_uri(r) for r in hk_graph.objects(uri, SKOS.related)
                  if isinstance(r, URIRef)]

    # Broader hierarchy
    broader_labels = [_label_for_uri(b) for b in hk_graph.objects(uri, SKOS.broader)
                      if isinstance(b, URIRef)]

    # Narrower children
    narrower_labels = [_label_for_uri(n) for n in hk_graph.objects(uri, SKOS.narrower)
                       if isinstance(n, URIRef)]

    # Regions (hk:occursIn)
    HK_OCCURS_IN = HK_NS.occursIn
    regions = [_label_for_uri(r) for r in hk_graph.objects(uri, HK_OCCURS_IN)
               if isinstance(r, URIRef)]

    # Discipline (hk:describedBy)
    HK_DESCRIBED_BY = HK_NS.describedBy
    disciplines = [_label_for_uri(d) for d in hk_graph.objects(uri, HK_DESCRIBED_BY)
                   if isinstance(d, URIRef)]

    # Principal flag
    is_principal = (uri, RDF.type, HK_PRINCIPAL_CLASS) in hk_graph

    term_norm = _normalize_term(pref_label)

    rows_hk.append({
        'hk_uri':        str(uri),
        'prefLabel':     pref_label,
        'term_norm':     term_norm,
        'altLabels':     ' | '.join(alt_labels),
        'hk_definition': definition,
        'broader':       ' | '.join(broader_labels),
        'narrower':      ' | '.join(narrower_labels),
        'regions':       ' | '.join(regions),
        'disciplines':   ' | '.join(disciplines),
        'related':       ' | '.join(related),
        'is_principal':  is_principal,
    })

df_hk = pd.DataFrame(rows_hk)
print(f'HelioKNOW phenomena table: {len(df_hk)} rows')
df_hk[['prefLabel', 'altLabels', 'hk_definition', 'broader', 'regions', 'is_principal']].head(20)

## 2  Explore HelioKNOW Phenomena Taxonomy

In [ ]:
# ── Top-level taxonomy overview ───────────────────────────────────────────────

# Phenomena with no broader (top-level within phenomena)
top_level = df_hk[df_hk['broader'] == ''][['prefLabel', 'narrower', 'regions', 'disciplines']]
print('Top-level HelioKNOW phenomena (no broader parent):')
print(top_level.to_string(index=False))

print(f'\nPrincipal phenomena:')
print(df_hk[df_hk['is_principal']][['prefLabel', 'broader', 'regions']].to_string(index=False))

In [ ]:
# ── Full phenomena list, sorted by broader category ────────────────────────────
print(f'All {len(df_hk)} HelioKNOW phenomena:\n')
for _, row in df_hk.sort_values(['broader', 'prefLabel']).iterrows():
    broader = f"  [{row['broader']}]" if row['broader'] else ''
    alt = f"  aka: {row['altLabels']}" if row['altLabels'] else ''
    print(f"  • {row['prefLabel']}{broader}{alt}")

## 3  Load All Glossaries and Extract Phenomenological Terms

Strategy: load every glossary, then **flag** entries as phenomenological based on:
1. Type annotation in the source (RDF type, JSON type field)
2. Hierarchy/path containing phenomenon keywords
3. Direct label match to a HelioKNOW phenomenon term or synonym
4. SWEET module membership (`phen*`, `proc*` modules)

In [ ]:
# ── Load standard JSON glossaries (all terms) ─────────────────────────────────

def _extract_definition(entry):
    direct = _first_nonempty(
        entry.get('Definition'), entry.get('definition'),
        entry.get('Description'), entry.get('description'),
    )
    if direct:
        return direct
    definitions = entry.get('definitions')
    if isinstance(definitions, list):
        texts = []
        for d in definitions:
            t = _first_nonempty(
                d.get('text') if isinstance(d, dict) else d,
                d.get('definition') if isinstance(d, dict) else None,
            )
            if t:
                texts.append(t)
        if texts:
            return ' | '.join(dict.fromkeys(texts))
    return None


def _extract_hierarchy(entry, term):
    hierarchy_fields = [
        'Hierarchy', 'hierarchy', 'Path', 'path', 'Parents', 'parents',
        'Broader', 'broader', 'broader_terms', 'categories', 'Category',
        'Topic', 'Term', 'Variable_Level_1', 'Variable_Level_2',
        'Variable_Level_3', 'Detailed_Variable',
    ]
    for key in hierarchy_fields:
        value = entry.get(key)
        if value is None:
            continue
        if isinstance(value, str):
            if '->' in value:
                parts = [p.strip() for p in value.split('->')]
                return _stringify_hierarchy(parts, term=term)
            if '/' in value:
                parts = [p.strip() for p in value.split('/')]
                return _stringify_hierarchy(parts, term=term)
            return _stringify_hierarchy([value], term=term)
        if isinstance(value, (list, tuple)):
            parts = []
            for item in value:
                if isinstance(item, dict):
                    parts.append(_first_nonempty(item.get('label'), item.get('name'), item.get('term')))
                else:
                    parts.append(item)
            return _stringify_hierarchy(parts, term=term)
    return term


def load_standard_glossaries(paths):
    rows = []
    for path in paths:
        if not os.path.exists(path):
            print(f'  [SKIP – file not found] {path}')
            continue
        source = Path(path).stem
        content = _safe_read_json(path)
        entries = []
        if isinstance(content, dict):
            entries = content.get('Terms') or content.get('terms') or content.get('concepts') or []
        elif isinstance(content, list):
            entries = content
        for entry in entries:
            if not isinstance(entry, dict):
                continue
            raw_term = _first_nonempty(
                entry.get('Term'), entry.get('term'), entry.get('prefLabel'),
                entry.get('label'), entry.get('name'),
            )
            term = _normalize_term(raw_term)
            if not term:
                continue
            definition  = _extract_definition(entry)
            hierarchy   = _extract_hierarchy(entry, term=raw_term)
            # Synonyms
            synonyms_raw = entry.get('Synonyms') or entry.get('synonyms') or entry.get('altLabel') or []
            if isinstance(synonyms_raw, str):
                synonyms_raw = [synonyms_raw]
            synonyms = [_normalize_term(s) for s in synonyms_raw if _normalize_term(s)]
            rows.append({
                'term':           term,
                'definition':     definition,
                'hierarchy_path': hierarchy,
                'synonyms':       synonyms,
                'source':         source,
            })
    return pd.DataFrame(rows)


print('Loading standard glossaries...')
df_standard = load_standard_glossaries(GLOSSARIES_LIST)
print(f'Standard glossaries: {len(df_standard):,} entries from {df_standard["source"].nunique()} sources')
df_standard['source'].value_counts()

In [ ]:
# ── Load GEMET ─────────────────────────────────────────────────────────────────

_SKOS  = Namespace('http://www.w3.org/2004/02/skos/core#')
_RDFS  = Namespace('http://www.w3.org/2000/01/rdf-schema#')
_DCTERMS = Namespace('http://purl.org/dc/terms/')

def load_gemet(gemet_paths):
    missing = [p for p in gemet_paths if not os.path.exists(p)]
    if missing:
        print(f'  [SKIP – GEMET files not found]: {missing}')
        return pd.DataFrame()

    graph = Graph()
    for path in gemet_paths:
        graph.parse(path)

    @lru_cache(maxsize=None)
    def pref_label(uri):
        labels = list(graph.objects(uri, _SKOS.prefLabel)) + list(graph.objects(uri, _RDFS.label))
        en = [v for v in labels if isinstance(v, Literal) and v.language in ('en', None)]
        sel = en[0] if en else (labels[0] if labels else None)
        return _clean_text(str(sel)) if sel else None

    @lru_cache(maxsize=None)
    def definition(uri):
        defs = (
            list(graph.objects(uri, _SKOS.definition))
            + list(graph.objects(uri, _DCTERMS.description))
            + list(graph.objects(uri, _RDFS.comment))
        )
        en = [v for v in defs if isinstance(v, Literal) and v.language in ('en', None)]
        sel = en[0] if en else (defs[0] if defs else None)
        return _clean_text(str(sel)) if sel else None

    @lru_cache(maxsize=None)
    def broader_nodes(uri):
        return [n for n in graph.objects(uri, _SKOS.broader) if isinstance(n, URIRef)]

    @lru_cache(maxsize=None)
    def parent_paths(uri, depth=0):
        if depth > 40:
            return [[]]
        parents = broader_nodes(uri)
        if not parents:
            return [[]]
        paths = []
        for parent in parents:
            for upstream in parent_paths(parent, depth + 1):
                if parent in upstream:
                    continue
                paths.append(upstream + [parent])
        seen = set()
        unique = []
        for p in paths:
            key = tuple(p)
            if key not in seen:
                seen.add(key)
                unique.append(p)
        return unique

    rows = []
    concepts = set(graph.subjects(_SKOS.prefLabel, None)) | set(graph.subjects(_RDFS.label, None))
    for concept in concepts:
        if not isinstance(concept, URIRef):
            continue
        raw_term = pref_label(concept)
        term = _normalize_term(raw_term)
        if not term:
            continue
        def_text = definition(concept)
        hierarchy_candidates = []
        for uri_path in parent_paths(concept):
            labels = [pref_label(n) for n in uri_path if pref_label(n)]
            h = _stringify_hierarchy(labels, term=raw_term)
            if h:
                hierarchy_candidates.append(h)
        hierarchy_candidates = sorted(set(hierarchy_candidates))
        hierarchy_path = ' | '.join(hierarchy_candidates) if hierarchy_candidates else raw_term
        rows.append({'term': term, 'definition': def_text, 'hierarchy_path': hierarchy_path,
                     'synonyms': [], 'source': 'GEMET'})
    return pd.DataFrame(rows)


print('Loading GEMET...')
df_gemet = load_gemet(GEMET_PATHS)
print(f'GEMET: {len(df_gemet):,} entries')

In [ ]:
# ── Load GCMD ──────────────────────────────────────────────────────────────────

def load_gcmd(gcmd_paths):
    csv_path = next((p for p in gcmd_paths if p.lower().endswith('.csv')), None)
    json_paths = [p for p in gcmd_paths if p.lower().endswith('.json')]
    if not csv_path or not os.path.exists(csv_path):
        print('  [SKIP – GCMD CSV not found]')
        return pd.DataFrame()

    hierarchy_by_uuid = {}
    term_by_uuid_csv = {}
    with open(csv_path, 'r', encoding='utf-8') as f:
        next(f)  # skip metadata line
        reader = csv.DictReader(f)
        for row in reader:
            uuid = _clean_text(row.get('UUID'))
            if not uuid:
                continue
            levels = [_clean_text(row.get(k)) for k in
                      ['Category', 'Topic', 'Term', 'Variable_Level_1',
                       'Variable_Level_2', 'Variable_Level_3', 'Detailed_Variable']
                      if _clean_text(row.get(k))]
            hierarchy_by_uuid[uuid] = _stringify_hierarchy(levels)
            if levels:
                term_by_uuid_csv[uuid] = levels[-1]

    concepts_by_uuid = {}
    for path in json_paths:
        if not os.path.exists(path):
            continue
        content = _safe_read_json(path)
        for concept in (content.get('concepts', []) if isinstance(content, dict) else []):
            if not isinstance(concept, dict):
                continue
            uuid = _clean_text(concept.get('uuid'))
            if not uuid:
                continue
            term = _first_nonempty(concept.get('prefLabel'), concept.get('label'))
            defs = concept.get('definitions', [])
            snippets = []
            if isinstance(defs, list):
                for d in defs:
                    t = _first_nonempty(
                        d.get('text') if isinstance(d, dict) else d,
                        d.get('definition') if isinstance(d, dict) else None,
                    )
                    if t:
                        snippets.append(t)
            concepts_by_uuid[uuid] = {
                'term': term,
                'definition': ' | '.join(dict.fromkeys(snippets)) if snippets else None,
            }

    rows = []
    for uuid in set(hierarchy_by_uuid) | set(concepts_by_uuid):
        info = concepts_by_uuid.get(uuid, {})
        raw_term = _first_nonempty(info.get('term'), term_by_uuid_csv.get(uuid))
        term = _normalize_term(raw_term)
        if not term:
            continue
        rows.append({
            'term': term,
            'definition': _clean_text(info.get('definition')),
            'hierarchy_path': hierarchy_by_uuid.get(uuid) or raw_term,
            'synonyms': [],
            'source': 'GCMD',
        })
    return pd.DataFrame(rows)


print('Loading GCMD...')
df_gcmd = load_gcmd(GCMD_PATHS)
print(f'GCMD: {len(df_gcmd):,} entries')

In [ ]:
# ── Load UAT (with definitions) ───────────────────────────────────────────────

def load_uat(uat_hierarchy_csv, uat_definitions_json):
    if not os.path.exists(uat_hierarchy_csv) or not os.path.exists(uat_definitions_json):
        print('  [SKIP – UAT files not found]')
        return pd.DataFrame()

    # Build definition map
    def_map = {}
    content = _safe_read_json(uat_definitions_json)
    def walk(node):
        if not isinstance(node, dict):
            return
        name = _clean_text(node.get('name'))
        defn = _clean_text(node.get('definition'))
        if name and defn and name.lower() not in def_map:
            def_map[name.lower()] = defn
        for child in (node.get('children') or []):
            walk(child)
    walk(content)

    uat_raw = pd.read_csv(uat_hierarchy_csv)
    level_cols = [c for c in uat_raw.columns if c.lower().startswith('level')]
    rows = []
    for _, row in uat_raw.iterrows():
        levels = [_clean_text(row[c]) for c in level_cols if _clean_text(row[c])]
        if not levels:
            continue
        term_display = levels[-1]
        term = _normalize_term(term_display)
        if not term:
            continue
        rows.append({
            'term': term,
            'definition': def_map.get(term_display.lower()),
            'hierarchy_path': _stringify_hierarchy(levels),
            'synonyms': [],
            'source': 'Unified Astronomy Thesaurus (UAT)',
        })
    return pd.DataFrame(rows)


print('Loading UAT...')
df_uat = load_uat(UAT_HIERARCHY_CSV, UAT_DEFINITIONS_JSON)
print(f'UAT: {len(df_uat):,} entries')

In [ ]:
# ── Load SWEET (helio/phenomenon-relevant modules) ────────────────────────────

def _normalize_module_name(m):
    m = m.strip()
    if m.lower().endswith('.ttl'):
        m = m[:-4]
    if m.startswith('http'):
        m = m.rstrip('/').rsplit('/', 1)[-1]
        if m.lower().endswith('.ttl'):
            m = m[:-4]
    return m


def load_sweet_modules(sweet_rdf_path, requested_modules):
    if not os.path.exists(sweet_rdf_path):
        print(f'  [SKIP – SWEET file not found: {sweet_rdf_path}]')
        return pd.DataFrame()

    graph = Graph()
    parsed = []
    failed = []
    t0 = time.perf_counter()
    for module in requested_modules:
        uri = f'http://sweetontology.net/{_normalize_module_name(module)}'
        try:
            graph.parse(uri)
            parsed.append(uri)
        except Exception as exc:
            failed.append((uri, str(exc)))

    print(f'  SWEET parsed {len(parsed)}/{len(requested_modules)} modules in {time.perf_counter()-t0:.1f}s')
    if failed:
        for uri, err in failed:
            print(f'    FAILED: {uri} -> {err}')

    @lru_cache(maxsize=None)
    def label(node):
        labels = list(graph.objects(node, RDFS.label))
        en = [v for v in labels if isinstance(v, Literal) and v.language in ('en', None)]
        sel = en[0] if en else (labels[0] if labels else None)
        if sel:
            return _clean_text(str(sel))
        tail = str(node).rsplit('/', 1)[-1].rsplit('#', 1)[-1]
        return _clean_text(tail)

    @lru_cache(maxsize=None)
    def parents(node):
        return [p for p in graph.objects(node, RDFS.subClassOf) if isinstance(p, URIRef)]

    @lru_cache(maxsize=None)
    def parent_paths(node, depth=0):
        if depth > 40:
            return [[]]
        pars = parents(node)
        if not pars:
            return [[]]
        built = []
        for parent in pars:
            for upstream in parent_paths(parent, depth + 1):
                if parent in upstream:
                    continue
                built.append(upstream + [parent])
        unique = []
        seen = set()
        for p in built:
            key = tuple(p)
            if key not in seen:
                seen.add(key)
                unique.append(p)
        return unique

    rows = []
    classes = set(graph.subjects(RDF.type, OWL.Class)) | set(graph.subjects(RDFS.subClassOf, None))
    for concept in classes:
        if not isinstance(concept, URIRef):
            continue
        raw_term = label(concept)
        term = _normalize_term(raw_term)
        if not term:
            continue
        definition_text = _literal_en(
            list(graph.objects(concept, RDFS.comment)) +
            list(graph.objects(concept, SKOS.definition))
        )
        hierarchy_candidates = []
        for path in parent_paths(concept):
            labels = [label(n) for n in path if label(n)]
            h = _stringify_hierarchy(labels, term=raw_term)
            if h:
                hierarchy_candidates.append(h)
        hierarchy_candidates = sorted(set(hierarchy_candidates))
        hierarchy_path = ' | '.join(hierarchy_candidates) if hierarchy_candidates else raw_term
        rows.append({
            'term': term,
            'definition': definition_text,
            'hierarchy_path': hierarchy_path,
            'synonyms': [],
            'source': 'SWEET',
        })
    return pd.DataFrame(rows)


print('Loading SWEET phenomenon modules...')
df_sweet = load_sweet_modules(SWEET_RDF_PATH, SWEET_PHEN_MODULES)
print(f'SWEET (phen modules): {len(df_sweet):,} entries')

In [ ]:
# ── Combine all glossary sources ──────────────────────────────────────────────

dfs_to_combine = [
    df for df in [df_standard, df_gemet, df_gcmd, df_uat, df_sweet]
    if df is not None and len(df) > 0
]

df_all_terms = pd.concat(dfs_to_combine, ignore_index=True)
df_all_terms = df_all_terms.drop_duplicates(
    subset=['term', 'definition', 'source']
).reset_index(drop=True)

print(f'Combined glossary: {len(df_all_terms):,} entries across {df_all_terms["source"].nunique()} sources')
df_all_terms['source'].value_counts()

## 4  Filter Glossary Entries: Phenomenon Candidates

An entry is a **phenomenon candidate** if any of the following are true:
- Its `term` or `hierarchy_path` matches a HelioKNOW phenomenon term or synonym
- Its `hierarchy_path` contains a phenomenon-indicative keyword
- It comes from a SWEET `phen*` or `proc*` module entry

In [ ]:
# ── Build lookup sets from HelioKNOW phenomena ─────────────────────────────────

# All normalized labels (prefLabel + altLabels) from HelioKNOW
hk_phen_terms = set()
for _, row in df_hk.iterrows():
    t = _normalize_term(row['prefLabel'])
    if t:
        hk_phen_terms.add(t)
    for alt in row['altLabels'].split(' | '):
        t2 = _normalize_term(alt)
        if t2:
            hk_phen_terms.add(t2)

print(f'HelioKNOW phenomenon lookup terms (prefLabel + altLabels): {len(hk_phen_terms)}')
print(sorted(hk_phen_terms)[:20], '...')

In [ ]:
# ── Flag phenomenon candidates ────────────────────────────────────────────────

def _is_phen_candidate(row):
    term = str(row.get('term', '')).lower()
    hierarchy = str(row.get('hierarchy_path', '') or '').lower()
    source = str(row.get('source', '')).lower()

    # Direct match to HelioKNOW phenomenon term
    if term in hk_phen_terms:
        return True, 'hk_match'

    # Hierarchy/term contains phenomenon keyword
    if _contains_phen_keyword(term + ' ' + hierarchy):
        return True, 'keyword_match'

    return False, None


flags = df_all_terms.apply(_is_phen_candidate, axis=1)
df_all_terms['is_phen_candidate'] = [f[0] for f in flags]
df_all_terms['phen_flag_reason']  = [f[1] for f in flags]

df_phen_candidates = df_all_terms[df_all_terms['is_phen_candidate']].copy()

print(f'Phenomenon candidates in glossaries: {len(df_phen_candidates):,}')
print(f'  of which have definitions: {df_phen_candidates["definition"].notna().sum():,}')
print(f'  from sources:')
print(df_phen_candidates['source'].value_counts().to_string())

## 5  Cross-Reference: What Is and Isn't in HelioKNOW?

Compare phenomena in HelioKNOW against all glossary phenomenon candidates to understand:
- Which HelioKNOW phenomena have definitions in other sources (coverage)
- Which phenomena appear in glossaries but are **not yet** in HelioKNOW (gaps)

In [ ]:
# ── For each HelioKNOW phenomenon, find matching glossary entries ──────────────

# Build per-term lookup from glossary phenomenon candidates
gloss_phen_by_term = {}
for _, row in df_phen_candidates.iterrows():
    t = row['term']
    gloss_phen_by_term.setdefault(t, []).append(row)


def _find_glossary_matches(hk_row):
    """Return list of (source, definition, matched_term) for a HelioKNOW phenomenon."""
    # Build set of normalized lookup keys: prefLabel + altLabels
    keys = set()
    t = _normalize_term(hk_row['prefLabel'])
    if t:
        keys.add(t)
    for alt in hk_row['altLabels'].split(' | '):
        t2 = _normalize_term(alt)
        if t2:
            keys.add(t2)

    matches = []
    for key in keys:
        for entry in gloss_phen_by_term.get(key, []):
            matches.append({
                'source':       entry['source'],
                'definition':   entry['definition'],
                'matched_term': key,
                'hierarchy':    entry['hierarchy_path'],
            })
    return matches


coverage_rows = []
for _, hk_row in df_hk.iterrows():
    matches = _find_glossary_matches(hk_row)
    n_sources = len({m['source'] for m in matches})
    n_defs = sum(1 for m in matches if m['definition'])
    coverage_rows.append({
        'hk_prefLabel':  hk_row['prefLabel'],
        'hk_definition': hk_row['hk_definition'],
        'broader':       hk_row['broader'],
        'regions':       hk_row['regions'],
        'altLabels':     hk_row['altLabels'],
        'is_principal':  hk_row['is_principal'],
        'n_glossary_sources': n_sources,
        'n_glossary_defs':    n_defs,
        'glossary_sources':   ' | '.join(sorted({m['source'] for m in matches})),
        '_matches':      matches,
    })

df_coverage = pd.DataFrame(coverage_rows)

print('HelioKNOW phenomena coverage across external glossaries:')
print(df_coverage[['hk_prefLabel', 'n_glossary_sources', 'n_glossary_defs', 'glossary_sources']]
      .sort_values('n_glossary_sources', ascending=False).to_string(index=False))

In [ ]:
# ── Coverage summary ──────────────────────────────────────────────────────────

with_ext_defs = (df_coverage['n_glossary_defs'] > 0).sum()
no_ext_defs   = (df_coverage['n_glossary_defs'] == 0).sum()

print(f"HelioKNOW phenomena: {len(df_coverage)}")
print(f"  With ≥1 external definition: {with_ext_defs}")
print(f"  No external definitions found: {no_ext_defs}")
print()
print('Phenomena lacking any external definition (candidates for enrichment):')
no_defs_list = df_coverage[df_coverage['n_glossary_defs'] == 0][['hk_prefLabel', 'hk_definition', 'broader']]
print(no_defs_list.to_string(index=False))

In [ ]:
# ── Glossary phenomena NOT in HelioKNOW ───────────────────────────────────────

# All normalized HK terms
hk_norm_terms = set(df_hk['term_norm'].dropna())
for _, row in df_hk.iterrows():
    for alt in row['altLabels'].split(' | '):
        t = _normalize_term(alt)
        if t:
            hk_norm_terms.add(t)

df_phen_not_in_hk = df_phen_candidates[
    ~df_phen_candidates['term'].isin(hk_norm_terms)
].copy()

# Summarize by unique term
df_gaps = (
    df_phen_not_in_hk
    .groupby('term', as_index=False)
    .agg(
        definitions=('definition', _unique_nonempty_list),
        sources=('source', lambda s: sorted(set(s))),
        hierarchy_paths=('hierarchy_path', _unique_nonempty_list),
    )
)
df_gaps['n_sources']     = df_gaps['sources'].map(len)
df_gaps['n_definitions'] = df_gaps['definitions'].map(len)
df_gaps = df_gaps.sort_values('n_sources', ascending=False).reset_index(drop=True)

print(f'Phenomena in glossaries but NOT in HelioKNOW: {len(df_gaps)} unique terms')
print(f'(top candidates by source coverage):')
df_gaps[['term', 'n_sources', 'n_definitions', 'sources']].head(40)

## 6  Build Unified Phenomena Table

Assemble one canonical row per phenomenon with every definition and source compiled.  
Canonical terms are drawn from HelioKNOW `prefLabel` (or the glossary term if not in HK).

In [ ]:
# ── Build long-form unified phenomena table ────────────────────────────────────
# Each row = one (canonical_term, source, definition) triple

long_rows = []

# --- Part A: HelioKNOW phenomena ---
for _, hk_row in df_hk.iterrows():
    canonical = hk_row['prefLabel']
    term_norm = hk_row['term_norm']

    # HelioKNOW native definition (if present)
    if hk_row['hk_definition']:
        long_rows.append({
            'canonical_term':  canonical,
            'term_norm':       term_norm,
            'source':          'HelioKNOW',
            'definition':      hk_row['hk_definition'],
            'hierarchy_path':  hk_row['broader'],
            'in_helioknow':    True,
            'hk_uri':          hk_row['hk_uri'],
            'hk_regions':      hk_row['regions'],
            'hk_disciplines':  hk_row['disciplines'],
            'hk_altLabels':    hk_row['altLabels'],
            'hk_is_principal': hk_row['is_principal'],
        })

    # External glossary matches
    matches = _find_glossary_matches(hk_row)
    for m in matches:
        if not m['definition']:
            continue
        long_rows.append({
            'canonical_term':  canonical,
            'term_norm':       term_norm,
            'source':          m['source'],
            'definition':      m['definition'],
            'hierarchy_path':  m['hierarchy'],
            'in_helioknow':    True,
            'hk_uri':          hk_row['hk_uri'],
            'hk_regions':      hk_row['regions'],
            'hk_disciplines':  hk_row['disciplines'],
            'hk_altLabels':    hk_row['altLabels'],
            'hk_is_principal': hk_row['is_principal'],
        })

# --- Part B: Glossary phenomena not in HelioKNOW ---
for _, row in df_phen_not_in_hk.iterrows():
    if not row['definition']:
        continue
    long_rows.append({
        'canonical_term':  row['term'],
        'term_norm':       row['term'],
        'source':          row['source'],
        'definition':      row['definition'],
        'hierarchy_path':  row['hierarchy_path'],
        'in_helioknow':    False,
        'hk_uri':          None,
        'hk_regions':      None,
        'hk_disciplines':  None,
        'hk_altLabels':    None,
        'hk_is_principal': False,
    })

df_long = pd.DataFrame(long_rows).drop_duplicates(
    subset=['canonical_term', 'source', 'definition']
).reset_index(drop=True)

print(f'Unified long-form phenomena table: {len(df_long):,} rows')
print(f'  In HelioKNOW: {df_long["in_helioknow"].sum():,} rows')
print(f'  Not in HelioKNOW: {(~df_long["in_helioknow"]).sum():,} rows')
df_long.head(10)

In [ ]:
# ── Build wide-form unified table (one row per canonical term) ─────────────────
# Columns: canonical_term | hk_definition | all_definitions (with sources) | sources | ...

def _format_def_with_source(rows_for_term):
    """Return list of 'definition  [source: X]' strings."""
    parts = []
    seen = set()
    for _, r in rows_for_term.iterrows():
        d = _clean_text(r['definition'])
        s = str(r['source']).strip()
        if not d:
            continue
        key = (d.lower()[:80], s)
        if key in seen:
            continue
        seen.add(key)
        parts.append(f"{d}  [source: {s}]")
    return parts


wide_rows = []
for canonical_term, grp in df_long.groupby('canonical_term', sort=True):
    in_hk  = bool(grp['in_helioknow'].any())
    hk_row = grp[grp['source'] == 'HelioKNOW'].iloc[0] if (grp['source'] == 'HelioKNOW').any() else None

    defs_sourced = _format_def_with_source(grp)
    all_sources  = sorted(grp['source'].dropna().unique())
    n_sources    = len(all_sources)
    n_defs       = len(defs_sourced)

    wide_rows.append({
        'canonical_term':      canonical_term,
        'in_helioknow':        in_hk,
        'hk_uri':              grp['hk_uri'].dropna().iloc[0] if grp['hk_uri'].notna().any() else None,
        'hk_altLabels':        grp['hk_altLabels'].dropna().iloc[0] if grp['hk_altLabels'].notna().any() else None,
        'hk_regions':          grp['hk_regions'].dropna().iloc[0] if grp['hk_regions'].notna().any() else None,
        'hk_disciplines':      grp['hk_disciplines'].dropna().iloc[0] if grp['hk_disciplines'].notna().any() else None,
        'hk_broader':          grp['hierarchy_path'].dropna().iloc[0] if (grp['source'] == 'HelioKNOW').any() and grp.loc[grp['source'] == 'HelioKNOW', 'hierarchy_path'].notna().any() else None,
        'hk_is_principal':     bool(grp['hk_is_principal'].any()),
        'n_sources':           n_sources,
        'n_definitions':       n_defs,
        'sources':             ' | '.join(all_sources),
        'definitions_and_sources': ' ;; '.join(defs_sourced),
    })

df_wide = pd.DataFrame(wide_rows)
df_wide = df_wide.sort_values(['in_helioknow', 'n_sources'], ascending=[False, False]).reset_index(drop=True)

print(f'Wide-form unified phenomena table: {len(df_wide)} rows')
print(f'  In HelioKNOW: {df_wide["in_helioknow"].sum()}')
print(f'  Not in HelioKNOW: {(~df_wide["in_helioknow"]).sum()}')
df_wide.head(20)

## 7  Coverage Analysis

In [ ]:
# ── Coverage heatmap: HelioKNOW phenomena × glossary sources ──────────────────

hk_terms_ordered = df_hk.sort_values(['broader', 'prefLabel'])['prefLabel'].tolist()
all_sources_in_long = sorted(df_long[df_long['in_helioknow']]['source'].unique())
# Exclude HelioKNOW itself from coverage matrix
ext_sources = [s for s in all_sources_in_long if s != 'HelioKNOW']

heatmap_data = []
for term in hk_terms_ordered:
    term_rows = df_long[
        (df_long['canonical_term'] == term) & (df_long['source'].isin(ext_sources))
    ]
    row = {'phenomenon': term}
    for src in ext_sources:
        has_def = bool((term_rows['source'] == src).any())
        row[src] = 1 if has_def else 0
    heatmap_data.append(row)

df_heatmap = pd.DataFrame(heatmap_data).set_index('phenomenon')
df_heatmap['TOTAL'] = df_heatmap.sum(axis=1)
df_heatmap = df_heatmap.sort_values('TOTAL', ascending=False)

print('Coverage matrix (1 = definition found in that source):')
df_heatmap

In [ ]:
# ── Source contribution summary ────────────────────────────────────────────────
import matplotlib.pyplot as plt

src_counts = df_long[df_long['in_helioknow']].groupby('source')['definition'].apply(
    lambda s: s.notna().sum()
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
src_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Definitions provided per source (for HelioKNOW phenomena)', fontsize=13)
ax.set_ylabel('Number of definitions')
ax.set_xlabel('Source')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()
print(src_counts)

In [ ]:
# ── Which existing harmonization terms are phenomena? ─────────────────────────
# Quick check against the large harmonization terms list if it exists

harm_path = Path(
    '/Users/ryanmc/Documents/Helio_ECIP/dev/Helio-KNOW/tools/glossary_harmonization/data/harmonization_terms_with_categories.csv'
)

if harm_path.exists():
    df_harm = pd.read_csv(harm_path)
    df_harm['term_norm'] = df_harm['term'].astype(str).str.strip().str.lower()

    # Mark terms that appear in HelioKNOW phenomena
    df_harm['is_hk_phenomenon'] = df_harm['term_norm'].isin(hk_phen_terms)

    # Mark terms that are phenomenon candidates (keyword match)
    def_col = 'definitions_and_fallback_hierarchy' if 'definitions_and_fallback_hierarchy' in df_harm.columns else 'definitions'
    df_harm['is_phen_keyword_match'] = (
        df_harm['term_norm'].apply(_contains_phen_keyword)
        | df_harm[def_col].fillna('').apply(_contains_phen_keyword)
    )

    phen_in_harm = df_harm[df_harm['is_hk_phenomenon'] | df_harm['is_phen_keyword_match']]
    print(f'Harmonization terms list: {len(df_harm)} total terms')
    print(f'  Direct HelioKNOW phenomenon match:  {df_harm["is_hk_phenomenon"].sum()}')
    print(f'  Phenomenon keyword match:            {df_harm["is_phen_keyword_match"].sum()}')
    print(f'  Union (phenomena in harmonization):  {len(phen_in_harm)}')
    print()
    print('HelioKNOW phenomena already in the harmonization list:')
    print(df_harm[df_harm['is_hk_phenomenon']]['term'].tolist())
    print()
#     harm_norm_terms = set(df_harm['term_norm'].dropna())
#     missing_from_harm = [
#         row['prefLabel'] for _, row in df_hk.iterrows()
#         if row['term_norm'] not in harm_norm_terms
#     ]
#     for t in sorted(missing_from_harm):
#         print(f'  • {t}')
# else:
#     print(f'Harmonization file not found at: {harm_path}')
#     print('Skipping cross-check with harmonization terms list.')
    
    print('HelioKNOW phenomena NOT YET in the harmonization list:')
    harm_norm_terms = set(df_harm['term_norm'].dropna())

    missing_from_harm = [
        row['prefLabel'] for _, row in df_hk.iterrows()
        if row['term_norm'] not in harm_norm_terms
    ]

    for t in sorted(missing_from_harm, key=lambda x: (x is None, str(x).lower())):
        print(f'  • {t}')
else:
    print(f'Harmonization file not found at: {harm_path}')
    print('Skipping cross-check with harmonization terms list.')

## 8  Export

In [ ]:
# ── Save outputs ──────────────────────────────────────────────────────────────

out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)

# 1. Long-form: one row per (canonical_term, source, definition)
long_path = out / 'phenomena_longform.csv'
df_long_save = df_long.drop(columns=['_matches'] if '_matches' in df_long.columns else []).copy()
df_long_save.to_csv(long_path, index=False, quoting=csv.QUOTE_ALL)
print(f'Saved long-form:    {long_path}  ({len(df_long_save):,} rows)')

# 2. Unified wide-form: one row per canonical phenomenon
unified_path = out / 'phenomena_unified.csv'
df_wide.to_csv(unified_path, index=False, quoting=csv.QUOTE_ALL)
print(f'Saved unified:      {unified_path}  ({len(df_wide):,} rows)')

# 3. HelioKNOW coverage summary
coverage_path = out / 'phenomena_helioknow_coverage.csv'
df_coverage_save = df_coverage.drop(columns=['_matches']).copy()
df_coverage_save.to_csv(coverage_path, index=False, quoting=csv.QUOTE_ALL)
print(f'Saved HK coverage:  {coverage_path}  ({len(df_coverage_save):,} rows)')

# 4. Gaps: glossary phenomena not in HelioKNOW
gaps_path = out / 'phenomena_gaps_not_in_helioknow.csv'
df_gaps['sources_str']     = df_gaps['sources'].apply(lambda s: ' | '.join(s))
df_gaps['definitions_str'] = df_gaps['definitions'].apply(lambda d: ' ;; '.join(d))
df_gaps[['term', 'n_sources', 'n_definitions', 'sources_str', 'definitions_str',
          'hierarchy_paths']].to_csv(gaps_path, index=False, quoting=csv.QUOTE_ALL)
print(f'Saved gaps list:    {gaps_path}  ({len(df_gaps):,} rows)')

print('\nAll outputs saved.')

## Summary

| Output | Description |
|--------|-------------|
| `phenomena_longform.csv` | One row per (phenomenon, source, definition) — full provenance trail |
| `phenomena_unified.csv` | One row per canonical phenomenon; all definitions & sources compiled |
| `phenomena_helioknow_coverage.csv` | HelioKNOW phenomena with cross-source coverage counts |
| `phenomena_gaps_not_in_helioknow.csv` | Phenomena in glossaries not yet in HelioKNOW — candidates for addition |